In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import randint, loguniform, uniform
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import joblib
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, MinMaxScaler, OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, precision_recall_curve, roc_curve, auc, average_precision_score
import utils
skew_data = FunctionTransformer(utils.skew_data_tr, validate=False)
clean_data = FunctionTransformer(utils.preprocess_tr, validate=False)

In [2]:
path = "gs://data-epip/train/paysim.parquet"
train_pdf = pd.read_parquet(path, engine='pyarrow').sample(1_000_000)


X_train, y_train = train_pdf.drop("isFraud", axis=1), train_pdf["isFraud"]
X_train.drop(["nameOrig", "nameDest", "step", "isFlaggedFraud"], axis=1, inplace=True)
num_cols = X_train.select_dtypes(include=["float", "int"]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["category", "str", "object"]).columns.tolist()

In [3]:
num_pipeline = Pipeline([
    (
        "skew",
        FunctionTransformer(
            utils.skew_data_tr,
            feature_names_out="one-to-one"
        )
    ),
    ("scaler", MinMaxScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            cat_cols
        )
    ],
    verbose_feature_names_out=False
)

preprocessor.set_output(transform="pandas")

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. ``""{feature_name}__{transformer_name}""``. See :meth:`str.format` method from the standard library for more info... versionadded:: 1.0.. versionchanged:: 1.6 `verbose_feature_names_out` can be a callable or a string to be formatted.",False
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The

In [4]:
# XgboostClassifier
# precision_recall vs roc_curve (TPR, FPR)

cv = StratifiedKFold(n_splits=3)

classifier = XGBClassifier(
    objective="binary:logistic",
    eval_metric="aucpr", # auprc
    tree_method="hist",
    random_state=42,
    n_jobs=1) # early_stopping_rounds=10

pipeline_xgb = ImbPipeline(
    steps=[
        ("clean_data", clean_data),
        ("preprocess", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", classifier)
    ]
)

param_distributions = {
    "classifier__n_estimators": randint(200, 1200),
    "classifier__learning_rate": loguniform(0.01, 0.3),
    "classifier__max_depth": randint(3, 11),
    "classifier__min_child_weight": randint(1, 15),
    "classifier__subsample": uniform(0.6, 0.4),
    "classifier__colsample_bytree": uniform(0.6, 0.4),
    "classifier__reg_alpha": loguniform(1e-5, 10),
    "classifier__reg_lambda": loguniform(1e-3, 100),
    "classifier__gamma": loguniform(1e-5, 10),
    "classifier__max_bin": randint(64, 512),
}


random_search = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_distributions,
    n_iter=1,
    scoring="average_precision",
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=3
)

In [5]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 1 candidates, totalling 3 fits
[CV] END classifier__colsample_bytree=0.749816047538945, classifier__gamma=5.061576888752303, classifier__learning_rate=0.1205712628744377, classifier__max_bin=252, classifier__max_depth=7, classifier__min_child_weight=7, classifier__n_estimators=321, classifier__reg_alpha=8.629132190071849e-05, classifier__reg_lambda=0.0019517224641449498, classifier__subsample=0.9464704583099741; total time= 1.3min
[CV] END classifier__colsample_bytree=0.749816047538945, classifier__gamma=5.061576888752303, classifier__learning_rate=0.1205712628744377, classifier__max_bin=252, classifier__max_depth=7, classifier__min_child_weight=7, classifier__n_estimators=321, classifier__reg_alpha=8.629132190071849e-05, classifier__reg_lambda=0.0019517224641449498, classifier__subsample=0.9464704583099741; total time= 1.3min
[CV] END classifier__colsample_bytree=0.749816047538945, classifier__gamma=5.061576888752303, classifier__learning_rate=0.12057126287

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...=None, ...))])"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__colsample_bytree': <scipy.stats....t 0x152e9dc50>, 'classifier__gamma': <scipy.stats....t 0x152e9ee50>, 'classifier__learning_rate': <scipy.stats....t 0x152e88750>, 'classifier__max_bin': <scipy.stats....t 0x152e9f410>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",1
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'average_precision'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",3
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo...shuffle=False)
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe est

In [6]:
path = "gs://data-epip/test/paysim.parquet"
train_pdf = pd.read_parquet(path, engine='pyarrow').sample(10000)


X_test, y_test = train_pdf.drop("isFraud", axis=1), train_pdf["isFraud"]
X_test.drop(["nameOrig", "nameDest", "step", "isFlaggedFraud"], axis=1, inplace=True)


In [7]:
print(classification_report(y_train, random_search.predict(X_train)))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00    998704
           1       0.12      1.00      0.22      1296

    accuracy                           0.99   1000000
   macro avg       0.56      0.99      0.61   1000000
weighted avg       1.00      0.99      0.99   1000000



In [8]:
print(classification_report(y_test, random_search.predict(X_test)))

              precision    recall  f1-score   support

           0       1.00      0.99      1.00      9988
           1       0.11      0.50      0.17        12

    accuracy                           0.99     10000
   macro avg       0.55      0.75      0.59     10000
weighted avg       1.00      0.99      1.00     10000



In [9]:
predict_proba = random_search.predict_proba(X_test)[:, 1]

In [10]:
fpr, tpr, thresholds = roc_curve(y_test, predict_proba)

print(auc(fpr, tpr))

0.9639400614070217


In [11]:
precision, recall, thresholds = precision_recall_curve(y_test, predict_proba)

print(auc(recall, precision))


0.5088608553138103


In [12]:
print(average_precision_score(y_test, predict_proba))

0.5094955732527842


In [13]:
joblib.dump(random_search.best_estimator_, '../output/xgboost-new-v2.pkl')

['../output/xgboost-new-v2.pkl']

In [31]:
# Faire fonctionner ma pipeline et lancer l'enseignement et de déploiement en ayant une idée du coût
# Comprendre la différence entre : 
# Précision recall curve (y_true / y_score)
# Roc curve  (y_true/y_socre)
# Auc_roc et auc pr (auprc)
# Average précision 
# Algorithme boosting et fonctionnement xgboost